In [1]:
"""
================================================================================
 QUADRATIC DISCRIMINANT ANALYSIS (QDA) -- EDA + FROM-SCRATCH + SKLEARN DEMO
================================================================================
Dataset : sklearn's Wine dataset (3 classes, 13 chemical-analysis features).
Wine is a great QDA showcase because the three cultivars genuinely have
different feature *variances* (not just different means) -- exactly the
situation where QDA's per-class covariance matrices beat LDA's single
shared covariance matrix.

Sections:
  1. Load data
  2. EDA            -- class balance, summary stats, correlation, pairplot,
                        per-class variance comparison (LDA vs QDA justification)
  3. From-scratch QDA implementing the derivation:
        delta_k(x) = -1/2 (x-mu_k)^T Sigma_k^-1 (x-mu_k) - 1/2 log|Sigma_k| + log(pi_k)
        y_hat      = argmax_k delta_k(x)
  4. Sklearn QDA (sanity check against the from-scratch version)
  5. Evaluation     -- accuracy, confusion matrix, classification report
  6. 2-D decision-boundary visualisation (LDA vs QDA side-by-side)
================================================================================
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import (
    QuadraticDiscriminantAnalysis,
    LinearDiscriminantAnalysis,
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

RNG = 42
PLOTS = "/home/claude/qda_demo/plots"
sns.set_theme(style="whitegrid", font_scale=1.0)
PALETTE = ["#1F5FB4", "#D1521F", "#2E8B57"]

# ==============================================================================
# 1. LOAD DATA
# ==============================================================================
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="cultivar")
class_names = data.target_names  # ['class_0', 'class_1', 'class_2']

df = X.copy()
df["cultivar"] = y.map(dict(enumerate(class_names)))

print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)
print(f"Shape: {X.shape[0]} samples x {X.shape[1]} features, {y.nunique()} classes")
print(df.groupby("cultivar").size().rename("count"))
print("\nFirst rows:\n", df.head(3))

# ==============================================================================
# 2. EXPLORATORY DATA ANALYSIS
# ==============================================================================

# --- 2.1 Class balance ---------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 4))
counts = df["cultivar"].value_counts().sort_index()
ax.bar(counts.index, counts.values, color=PALETTE)
ax.set_title("Class Balance")
ax.set_ylabel("Number of samples")
for i, v in enumerate(counts.values):
    ax.text(i, v + 1, str(v), ha="center", fontweight="bold")
fig.tight_layout()
fig.savefig(f"{PLOTS}/01_class_balance.png", dpi=160)
plt.close(fig)

# --- 2.2 Summary statistics per class ------------------------------------
summary = df.groupby("cultivar").agg(["mean", "std"]).round(2)
print("\n" + "=" * 70)
print("PER-CLASS MEAN / STD (first 4 features)")
print("=" * 70)
print(summary.iloc[:, :8])

# --- 2.3 Correlation heatmap ----------------------------------------------
fig, ax = plt.subplots(figsize=(9, 7.5))
corr = X.corr()
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, ax=ax,
            cbar_kws={"shrink": 0.8}, linewidths=0.3)
ax.set_title("Feature Correlation Matrix (all classes pooled)")
fig.tight_layout()
fig.savefig(f"{PLOTS}/02_correlation_heatmap.png", dpi=160)
plt.close(fig)

# --- 2.4 Pairplot of the most discriminating features ---------------------
# pick 4 features with the largest between-class mean spread (ANOVA-F-like)
feat_scores = {}
for col in X.columns:
    grp_means = df.groupby("cultivar")[col].mean()
    feat_scores[col] = grp_means.std() / (X[col].std() + 1e-9)
top_feats = sorted(feat_scores, key=feat_scores.get, reverse=True)[:4]
print("\nTop 4 most discriminating features:", top_feats)

g = sns.pairplot(df[top_feats + ["cultivar"]], hue="cultivar",
                  palette=PALETTE, corner=True, diag_kind="kde",
                  plot_kws={"alpha": 0.7, "s": 30})
g.fig.suptitle("Pairplot of Top Discriminating Features", y=1.02, fontweight="bold")
g.fig.savefig(f"{PLOTS}/03_pairplot_top_features.png", dpi=160, bbox_inches="tight")
plt.close("all")

# --- 2.5 Per-class variance comparison: WHY QDA over LDA here -------------
# This is the key EDA plot for justifying QDA: if classes had equal
# covariance, these bars would be flat/equal per feature. They aren't.
std_by_class = df.groupby("cultivar")[top_feats].std()
fig, axes = plt.subplots(1, len(top_feats), figsize=(3.1 * len(top_feats), 4.4), sharey=False)
for ax, feat in zip(axes, top_feats):
    vals = std_by_class[feat]
    ax.bar(vals.index, vals.values, color=PALETTE)
    ax.set_title(feat, fontsize=11)
    ax.set_ylabel("Std. dev.")
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("Per-Class Std. Dev. of Top Features (own scale per feature)\n"
             "Unequal spread across cultivars \u2192 QDA is the better-justified model",
             fontweight="bold", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.86])
fig.savefig(f"{PLOTS}/04_variance_by_class.png", dpi=160)
plt.close(fig)

# --- 2.6 Covariance-matrix heatmaps per class (2 features) ----------------
two_feats = top_feats[:2]
short_labels = ["flavanoids", "OD280/OD315"]  # readable short axis labels
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for i, cname in enumerate(class_names):
    sub = X[y == i][two_feats]
    cov = sub.cov().values
    sns.heatmap(cov, annot=True, fmt=".2f", cmap="Blues", ax=axes[i],
                cbar=False, square=True, xticklabels=short_labels,
                yticklabels=short_labels, annot_kws={"size": 11})
    axes[i].set_title(cname, fontsize=12)
    axes[i].tick_params(axis="x", rotation=20)
    axes[i].tick_params(axis="y", rotation=0)
fig.suptitle("Class-Specific Covariance Matrices Differ \u2192 QDA Assumption Holds", fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.86])
fig.savefig(f"{PLOTS}/05_covariance_per_class.png", dpi=160)
plt.close(fig)

print("\nEDA plots written to:", PLOTS)

# ==============================================================================
# 3. FROM-SCRATCH QDA (implements the exact derivation from the write-up)
# ==============================================================================
class QDAFromScratch:
    """
    Quadratic Discriminant Analysis, implemented directly from:

        delta_k(x) = -1/2 (x-mu_k)^T Sigma_k^-1 (x-mu_k)
                     - 1/2 log|Sigma_k| + log(pi_k)

        y_hat = argmax_k delta_k(x)

    and posterior probabilities recovered via softmax over delta_k(x).
    """

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n, p = X.shape

        self.priors_ = {}
        self.means_ = {}
        self.covs_ = {}
        self.cov_inv_ = {}
        self.cov_logdet_ = {}

        for k in self.classes_:
            Xk = X[y == k]
            n_k = Xk.shape[0]
            self.priors_[k] = n_k / n
            mu_k = Xk.mean(axis=0)
            self.means_[k] = mu_k
            # unbiased sample covariance (n_k - 1 denominator), matches MLE-plug-in convention used in sklearn
            cov_k = np.cov(Xk, rowvar=False, ddof=1)
            # tiny ridge for numerical stability (near-singular covariances)
            cov_k = cov_k + 1e-6 * np.eye(p)
            self.covs_[k] = cov_k
            self.cov_inv_[k] = np.linalg.inv(cov_k)
            sign, logdet = np.linalg.slogdet(cov_k)
            self.cov_logdet_[k] = logdet
        return self

    def _delta(self, X):
        """Return an (n_samples, n_classes) matrix of delta_k(x) scores."""
        X = np.asarray(X, dtype=float)
        n = X.shape[0]
        K = len(self.classes_)
        deltas = np.zeros((n, K))
        for j, k in enumerate(self.classes_):
            diff = X - self.means_[k]                       # (n, p)
            maha = np.einsum("ij,jk,ik->i", diff, self.cov_inv_[k], diff)  # Mahalanobis^2 per row
            deltas[:, j] = (
                -0.5 * maha
                - 0.5 * self.cov_logdet_[k]
                + np.log(self.priors_[k])
            )
        return deltas

    def predict(self, X):
        deltas = self._delta(X)
        idx = np.argmax(deltas, axis=1)
        return self.classes_[idx]

    def predict_proba(self, X):
        deltas = self._delta(X)
        # softmax, shifted for numerical stability
        deltas -= deltas.max(axis=1, keepdims=True)
        exp = np.exp(deltas)
        return exp / exp.sum(axis=1, keepdims=True)


# ==============================================================================
# 4. TRAIN / TEST SPLIT + FIT BOTH VERSIONS
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.3, random_state=RNG, stratify=y.values
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

my_qda = QDAFromScratch().fit(X_train_s, y_train)
sk_qda = QuadraticDiscriminantAnalysis().fit(X_train_s, y_train)
sk_lda = LinearDiscriminantAnalysis().fit(X_train_s, y_train)  # for comparison

my_pred = my_qda.predict(X_test_s)
sk_pred = sk_qda.predict(X_test_s)
lda_pred = sk_lda.predict(X_test_s)

print("\n" + "=" * 70)
print("SANITY CHECK: from-scratch QDA vs sklearn QDA")
print("=" * 70)
agreement = (my_pred == sk_pred).mean()
print(f"Prediction agreement between from-scratch and sklearn QDA: {agreement:.4f}")
print(f"From-scratch QDA accuracy : {accuracy_score(y_test, my_pred):.4f}")
print(f"Sklearn QDA accuracy      : {accuracy_score(y_test, sk_pred):.4f}")
print(f"Sklearn LDA accuracy      : {accuracy_score(y_test, lda_pred):.4f}  (comparison)")

# ==============================================================================
# 5. EVALUATION
# ==============================================================================
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT (from-scratch QDA)")
print("=" * 70)
print(classification_report(y_test, my_pred, target_names=class_names))

cm = confusion_matrix(y_test, my_pred)
fig, ax = plt.subplots(figsize=(5, 4.3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=class_names, yticklabels=class_names, cbar=False)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix -- From-Scratch QDA (test set)")
fig.tight_layout()
fig.savefig(f"{PLOTS}/06_confusion_matrix.png", dpi=160)
plt.close(fig)

# ==============================================================================
# 6. 2-D DECISION BOUNDARY: LDA vs QDA (using the two most discriminating features)
# ==============================================================================
f1, f2 = top_feats[0], top_feats[1]
X2 = X[[f1, f2]].values
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y.values, test_size=0.3, random_state=RNG, stratify=y.values
)
scaler2 = StandardScaler().fit(X2_train)
X2_train_s = scaler2.transform(X2_train)
X2_test_s = scaler2.transform(X2_test)

qda2 = QuadraticDiscriminantAnalysis().fit(X2_train_s, y2_train)
lda2 = LinearDiscriminantAnalysis().fit(X2_train_s, y2_train)

xx, yy = np.meshgrid(
    np.linspace(X2_train_s[:, 0].min() - 1, X2_train_s[:, 0].max() + 1, 400),
    np.linspace(X2_train_s[:, 1].min() - 1, X2_train_s[:, 1].max() + 1, 400),
)
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.3))
for ax, (name, model) in zip(axes, [("LDA", lda2), ("QDA", qda2)]):
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.35, colors=PALETTE, levels=[-0.5, 0.5, 1.5, 2.5])
    for cls, color in zip(np.unique(y2_train), PALETTE):
        pts = X2_train_s[y2_train == cls]
        ax.scatter(pts[:, 0], pts[:, 1], s=22, color=color, edgecolor="white",
                   linewidth=0.4, label=class_names[cls])
    acc = accuracy_score(y2_test, model.predict(X2_test_s))
    ax.set_title(f"{name} decision regions\n(2-feature test acc = {acc:.2f})")
    ax.set_xlabel(f1)
    if ax is axes[0]:
        ax.set_ylabel(f2)
        ax.legend(fontsize=9, loc="best")
fig.suptitle(f"LDA vs QDA Decision Boundaries  ({f1}  vs  {f2})", fontweight="bold", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(f"{PLOTS}/07_lda_vs_qda_boundary.png", dpi=160)
plt.close(fig)

print("\nAll plots saved in:", PLOTS)
print("Done.")


DATASET OVERVIEW
Shape: 178 samples x 13 features, 3 classes
cultivar
class_0    59
class_1    71
class_2    48
Name: count, dtype: int64

First rows:
    alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   

   od280/od315_of_diluted_wines  proline cultivar  
0                          3.92   1065.0  class_0  
1                          3.40   1050.0  class_0  
2                          3.17   1185.0  class_0  


FileNotFoundError: [Errno 2] No such file or directory: '/home/claude/qda_demo/plots/01_class_balance.png'